# AutoSolve ML Training Pipeline

This notebook trains and exports the machine learning models (`track_predictor.onnx`, `settings_model.onnx`, and `region_weights.json`) for the **AutoSolve** Blender camera tracking addon.

### 🔁 Workflow Overview:
1. **Phase 1: Deep Training (No Blender Needed)**:
   * Upload video clips to `ml/clips/`.
   * Run OpenCV/CoTracker scripts natively to extract video features and point tracking trajectories.
   * Train models and export ONNX files entirely in this notebook.
2. **Phase 2: Validation (Blender Needed)**:
   * Run validation solves headlessly in Blender locally or in Colab (Step 8) to verify solve metrics.

---
## Google Drive Persistent Storage (Recommended for Colab)
Enable this option to link directory structures directly to your Google Drive (`My Drive/AutoSolve_ML_Data`), allowing you to persist video clips, uploaded logs, and training checkpoints.

In [ ]:
# @title Configure Google Drive Integration
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
DRIVE_PROJECT_PATH = "AutoSolve_ML_Data" # @param {type:"string"}

import os
import shutil

in_colab = False
try:
    import google.colab
    in_colab = True
except ImportError:
    pass

if in_colab and USE_GOOGLE_DRIVE:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    persist_dir = f"/content/drive/MyDrive/{DRIVE_PROJECT_PATH}"
    os.makedirs(persist_dir, exist_ok=True)
    
    # Create persistent subdirectories
    persistent_clips_dir = os.path.join(persist_dir, "clips")
    persistent_data_dir = os.path.join(persist_dir, "data")
    persistent_runs_dir = os.path.join(persist_dir, "runs")
    
    for d in [persistent_clips_dir, persistent_data_dir, persistent_runs_dir]:
        os.makedirs(d, exist_ok=True)
        
    print(f"\n📂 Google Drive paths mapped:")
    print(f"   Clips folder:    {persistent_clips_dir}")
    print(f"   Datasets folder: {persistent_data_dir}")
    print(f"   Runs folder:     {persistent_runs_dir}")
    
    # Setup symlinks in workspace
    for path, target in [("ml/clips", persistent_clips_dir), ("ml/data", persistent_data_dir), ("ml/runs", persistent_runs_dir)]:
        if os.path.exists(path):
            if os.path.islink(path): os.unlink(path)
            elif os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
        os.symlink(target, path)
        print(f"✅ Linked '{path}' -> '{target}'")
else:
    print("Using temporary local filesystem inside notebook environment.")
    for d in ['ml/clips', 'ml/data', 'ml/runs', 'ml/data/raw', 'ml/data/processed', 'ml/data/live']:
        os.makedirs(d, exist_ok=True)
    print("👉 Upload your video clips directly to 'ml/clips/' and solve logs to 'ml/data/raw/'.")

# Option A: clone from GitHub
# !git clone https://github.com/usamasq/AutoSolve.git
# %cd AutoSolve

# Install python dependencies
!pip install -q torch numpy onnx onnxruntime opencv-python
# Optional: install CoTracker for dense deep learning point tracking (falls back to OpenCV LK if not installed)
# !pip install git+https://github.com/facebookresearch/co-tracker.git

import torch, numpy as np, onnx, onnxruntime as ort
print(f"PyTorch:     {torch.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"ONNX:        {onnx.__version__}")
print(f"onnxruntime: {ort.__version__}")
print(f"GPU (CUDA):  {torch.cuda.is_available()}")

In [ ]:
---
## Step 2: Feature Ingestion and Dataset Preparation
Extract features from clips and generate tracking datasets natively using PyTorch and OpenCV.

# 1. Extract visual features (motion, zoom, distortion) directly from raw clips in ml/clips/
!python ml/extract_video_features.py --clips-dir ml/clips --out-dir ml/data/raw

In [ ]:
# 2. Extract point trajectories and simulate settings variations (with noise/occlusions)
!python ml/extract_cotracker_trajectories.py --clips-dir ml/clips --out-dir ml/data/raw

In [ ]:
# 3. Compile all raw logs into settings_dataset.json using the heuristic video features simulator
!python ml/prepare_dataset.py \
    --data-dir ml/data/raw \
    --output-json ml/data/processed/settings_dataset.json

In [ ]:
# 3. Compile all raw logs into settings_dataset.json
!python ml/prepare_dataset.py \
    --data-dir ml/data/raw \
    --output-json ml/data/processed/settings_dataset.json

---
## Step 3: Train Track Quality Predictor
Trains the neural network that predicts if an active track is likely to fail in the next 20 frames.

In [ ]:
# @title Track Quality Predictor Training Configuration
EPOCHS = 100 # @param {type:"integer"}

# @markdown Leave PRETRAINED_TRACK_MODEL blank ("") to train a new model from scratch.
# @markdown If you want to resume training, enter path to weights (e.g. ml/runs/track_predictor/model_meta_weights.json)
PRETRAINED_TRACK_MODEL = "" # @param {type:"string"}

cmd = f"python ml/train_track_predictor.py --data-dir ml/data/raw --out-dir ml/runs/track_predictor --epochs {EPOCHS}"
if PRETRAINED_TRACK_MODEL.strip():
    cmd += f" --pretrained \"{PRETRAINED_TRACK_MODEL.strip()}\""
    
print(f"Running command: {cmd}")
!{cmd}

---
## Step 4: Train Settings Reward Model
Trains the expected-reward model predicting how well a set of tracking parameters will perform given footage characteristics.

In [ ]:
# @title Settings Reward Model Configuration
EPOCHS = 50 # @param {type:"integer"}

# @markdown Leave PRETRAINED_SETTINGS_MODEL blank ("") to train a new model from scratch.
# @markdown If you want to resume training, enter path to weights (e.g. ml/runs/settings_optimizer/model_meta_weights.json)
PRETRAINED_SETTINGS_MODEL = "" # @param {type:"string"}

cmd = f"python ml/train_settings_model.py --data-path ml/data/processed/settings_dataset.json --out-dir ml/runs/settings_optimizer --epochs {EPOCHS}"
if PRETRAINED_SETTINGS_MODEL.strip():
    cmd += f" --pretrained \"{PRETRAINED_SETTINGS_MODEL.strip()}\""
    
print(f"Running command: {cmd}")
!{cmd}

In [ ]:
# Evaluate prediction metrics on validation split
!python ml/evaluate_model.py \
    --data-path ml/data/processed/settings_dataset.json \
    --model-path ml/runs/settings_optimizer/model_meta_weights.json

---
## Step 5: Aggregate Region Trackability Heatmap
Aggregates coordinates to find which areas of the screen produce the best-surviving tracking markers.

In [ ]:
!python ml/train_trackability_model.py \
    --data-dir ml/data/raw \
    --output-json ml/runs/region_weights.json

---
## Step 6: Export Models
Converts the trained models to standard **ONNX** formats for fast inference, and **JSON** fallback formats.

In [ ]:
# Export trained PyTorch weights to ONNX format
!python ml/export_onnx.py \
    --track-weights ml/runs/track_predictor/model_meta_weights.json \
    --settings-weights ml/runs/settings_optimizer/model_meta_weights.json \
    --out-dir ml/runs/onnx

In [ ]:
# Also export JSON/NumPy fallback weights
!python ml/export_numpy_model.py \
    --input-json ml/runs/track_predictor/model_meta_weights.json \
    --output-path ml/runs/track_predictor.json

!python ml/export_defaults.py \
    --model-path ml/runs/settings_optimizer/model_meta_weights.json \
    --output-json ml/runs/recommended_defaults.json

In [ ]:
# Verification: Output files status check
import os

expected = [
    'ml/runs/onnx/track_predictor.onnx',
    'ml/runs/onnx/track_predictor_meta.json',
    'ml/runs/onnx/settings_model.onnx',
    'ml/runs/onnx/settings_model_meta.json',
    'ml/runs/track_predictor.json',
    'ml/runs/region_weights.json',
    'ml/runs/recommended_defaults.json',
]

all_ok = True
print("Trained Model Output Verification:\n")
for f in expected:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) // 1024 if exists else 0
    status = f'✅  {size:4d} KB' if exists else '❌  MISSING'
    print(f'{status}   {f}')
    if not exists: all_ok = False

---
## Step 7: Download and Deploy Models to Blender Addon

Run the cell below to download your trained models, then copy them into your local **`autosolve/`** addon folder:

| Exported File | Target Addon Destination Path |
|---|---|
| `track_predictor.onnx` | `autosolve/tracker/models/track_predictor.onnx` |
| `track_predictor_meta.json` | `autosolve/tracker/models/track_predictor_meta.json` |
| `settings_model.onnx` | `autosolve/tracker/models/settings_model.onnx` |
| `settings_model_meta.json` | `autosolve/tracker/models/settings_model_meta.json` |
| `track_predictor.json` | `autosolve/tracker/models/track_predictor.json` *(fallback)* |
| `region_weights.json` | `autosolve/tracker/presets/region_weights.json` |
| `recommended_defaults.json` | Merge into `PRETRAINED_DEFAULTS` inside `autosolve/tracker/constants.py` |

In [ ]:
# Download output files (Colab environment only)
import os
try:
    from google.colab import files
    to_download = [
        'ml/runs/onnx/track_predictor.onnx',
        'ml/runs/onnx/track_predictor_meta.json',
        'ml/runs/onnx/settings_model.onnx',
        'ml/runs/onnx/settings_model_meta.json',
        'ml/runs/track_predictor.json',
        'ml/runs/region_weights.json',
        'ml/runs/recommended_defaults.json',
    ]
    print("Downloading files directly to browser:")
    for f in to_download:
        if os.path.exists(f):
            print(f'  Downloading: {f}')
            files.download(f)
except ImportError:
    print("Running locally. Models are located in your local project 'ml/runs/' folder.")

---
## Step 8 (Optional): Cloud-Based Headless Simulation & Validation
Use this section if you want to run camera tracking simulations using headless Blender directly inside Google Colab (requires downloading Blender Linux binaries).

In [ ]:
# 1. Download and extract Blender 5.1 for Linux
import os
if not os.path.exists("blender-5.1.0-linux-x64"):
    print("Downloading Blender 5.1.0...")
    !wget -q https://download.blender.org/release/Blender5.1/blender-5.1.0-linux-x64.tar.xz
    print("Extracting Blender...")
    !tar -xf blender-5.1.0-linux-x64.tar.xz
    !rm blender-5.1.0-linux-x64.tar.xz
    print("Blender ready!")
else:
    print("Blender already downloaded.")

In [ ]:
# 2. Run headless solve simulations over all clips in ml/clips/
!python ml/run_collection.py --clips-dir ml/clips --output-dir ml/data/raw --blender-bin ./blender-5.1.0-linux-x64/blender

In [ ]:
# 3. Deploy models locally to test validation sweeps
import os, shutil

os.makedirs("autosolve/tracker/models", exist_ok=True)
os.makedirs("autosolve/tracker/presets", exist_ok=True)

copies = [
    ("ml/runs/onnx/track_predictor.onnx",      "autosolve/tracker/models/track_predictor.onnx"),
    ("ml/runs/onnx/track_predictor_meta.json", "autosolve/tracker/models/track_predictor_meta.json"),
    ("ml/runs/onnx/settings_model.onnx",       "autosolve/tracker/models/settings_model.onnx"),
    ("ml/runs/onnx/settings_model_meta.json",  "autosolve/tracker/models/settings_model_meta.json"),
    ("ml/runs/region_weights.json",            "autosolve/tracker/presets/region_weights.json"),
    ("ml/runs/track_predictor.json",           "autosolve/tracker/models/track_predictor.json")
]

for src, dst in copies:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Deployed: {src} -> {dst}")

In [ ]:
# 4. Run validation sweeps on clips using newly deployed models
!python ml/run_collection.py --clips-dir ml/clips --output-dir ml/data/validation --blender-bin ./blender-5.1.0-linux-x64/blender

In [ ]:
# 5. Print out final solve statistics
import os, json

val_dir = "ml/data/validation"
if os.path.exists(val_dir):
    val_files = [f for f in os.listdir(val_dir) if f.endswith(".json")]
    print(f"Validation sweeps complete: {len(val_files)} samples.\n")
    for f in val_files:
        try:
            with open(os.path.join(val_dir, f)) as fh:
                data = json.load(fh)
                clip = data.get("clip_metadata", {}).get("clip_name", "Unknown")
                success = data.get("solve_success", False)
                error = data.get("solve_error", 99.0)
                ratio = data.get("bundle_ratio", 0.0)
                status = f"✅ Success ({error:.2f}px error, {ratio*100:.1f}% bundle ratio)" if success else "❌ Solve Failed"
                print(f"  {clip:30s} | {status}")
        except Exception as e:
            print(f"Error reading {f}: {e}")
else:
    print("Validation outputs directory not found.")